# ECIS — QLoRA fine-tune

Loads Llama 3.1 8B in NF4, attaches rank-16 LoRA on `q_proj` / `k_proj` / `v_proj` / `o_proj`, and trains with TRL `SFTTrainer`.

Training data is the JSONL from:

```bash
python -m ecis.main --export-qlora data/qlora/maintained_none.jsonl
```

Runtime → Change runtime type → GPU (A100). Checkpoints write to Google Drive every 100 steps.

In [ ]:
%pip install -q unsloth trl bitsandbytes datasets accelerate peft

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/ecis_qlora")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = Path("/content/maintained_none.jsonl")
print("Checkpoint dir:", DRIVE_DIR)

## Upload JSONL

In [ ]:
import json
from datasets import Dataset

assert JSONL_PATH.exists(), "Upload maintained_none.jsonl to /content/ first"
rows = [json.loads(line) for line in JSONL_PATH.read_text().splitlines() if line.strip()]
print(f"Loaded {len(rows)} triples")

def to_text(row):
    return (
        f"### Instruction:\n{row['instruction']}\n\n"
        f"### Input:\n{row['input']}\n\n"
        f"### Response:\n{row['output']}"
    )

dataset = Dataset.from_list([{"text": to_text(r)} for r in rows])
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds, eval_ds)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Adapter attached. Trainable params:", model.print_trainable_parameters())

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

args = TrainingArguments(
    output_dir=str(DRIVE_DIR / "checkpoints"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    args=args,
)
trainer.train()
print("Training loss:", trainer.state.log_history[-1])

In [ ]:
adapter_dir = DRIVE_DIR / "llama3.1-ecis-ft"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved adapter to", adapter_dir)
print("Serve with Ollama after merge, then set LLM_FINETUNED_MODEL=llama3.1-ecis-ft:latest")